In [3]:
# Run in Jupyter: the bounds and substitutions are rendered as display math.
# Dependencies: Python 3.10+, SymPy, IPython.
# All shared parent-search functions below are copied verbatim from
# parent_search_common.py. No import from that file is required.
# The empty word is ''. Templates are (boundary_letters, difference_vectors).
# All four examples use both-sieved ANCESTORS, including the zero template.

from sympy import Matrix, Rational, ceiling
from IPython.display import display, Math, Markdown
from itertools import product
from collections import deque, Counter
from math import lcm, isqrt


# ==================== Original parent-search functions: unchanged ====================

def parikh(word, alphabet):
    return tuple(word.count(a) for a in alphabet)


def frequency_matrix(h, alphabet):
    return Matrix([parikh(h[a], alphabet) for a in alphabet])


def make_template(A, D):
    return tuple(A), tuple(tuple(d) for d in D)


def zero_template(k, alphabet):
    return make_template(['']*(k+1), [(0,)*len(alphabet)]*(k-1))


def print_morphism(h, k):
    images = [rf'{a}\mapsto\mathtt{{{word}}}' for a, word in h.items()]
    display(Math(r'h:\quad ' + r',\quad '.join(images) + rf',\qquad k={k}'))
    print()


def split_set(h, a, nontrivial=False):
    splits = [('', '', '', '')] if a == '' else []
    for A, word in h.items():
        if a == '':
            for j in range(len(word)+1):
                if nontrivial and (j == 0 or j == len(word)):
                    continue
                splits.append((A, word[:j], '', word[j:]))
        else:
            for j, letter in enumerate(word):
                if letter == a:
                    splits.append((A, word[:j], a, word[j+1:]))
    return tuple(splits)


def split_sets(h, alphabet, nontrivial=False):
    return {
        a: tuple((A, parikh(p, alphabet), parikh(s, alphabet))
                 for A, p, middle, s in split_set(h, a, nontrivial))
        for a in ('',)+tuple(alphabet)
    }


def normalize_score(v, moduli):
    return tuple(x % q if q else x for x, q in zip(v, moduli))


def canonical_score(v, alphabet, weights, moduli):
    return normalize_score(
        tuple(sum(v[i]*weights[a][j] for i, a in enumerate(alphabet))
              for j in range(len(moduli))),
        moduli
    )


def score_buckets(S, alphabet, weights, moduli):
    prefixes = {p for splits in S.values() for A, p, s in splits}
    prefix_scores = {p: canonical_score(p, alphabet, weights, moduli)
                     for p in prefixes}
    B = {}
    for a, splits in S.items():
        B[a] = {}
        for split in splits:
            B[a].setdefault(prefix_scores[split[1]], []).append(split)
    return B


def nontrivial_buckets(B):
    B1 = {a: dict(buckets) for a, buckets in B.items()}
    B1[''] = {}
    for q, splits in B[''].items():
        remaining = tuple(split for split in splits
                          if split[0] == '' or (any(split[1]) and any(split[2])))
        if remaining:
            B1[''][q] = remaining
    return B1


def admissible_score_sequences(t, B, alphabet, weights, moduli):
    A, D = t
    k = len(A)-1
    d_scores = [canonical_score(d, alphabet, weights, moduli) for d in D]
    for q0 in B[A[0]]:
        for q1 in B[A[1]]:
            scores = [q0, q1]
            for i in range(1, k):
                q = normalize_score(
                    tuple(2*scores[i][j]-scores[i-1][j]
                          - weights[A[i-1]][j]+weights[A[i]][j]+d_scores[i-1][j]
                          for j in range(len(moduli))),
                    moduli
                )
                if q not in B[A[i+1]]:
                    break
                scores.append(q)
            else:
                yield tuple(scores)


def prepare_inverse(M):
    inverse = M.inv()
    denominator = lcm(*(int(x.q) for x in inverse))
    N = tuple(tuple(int(denominator*inverse[i, j]) for j in range(M.cols))
              for i in range(M.rows))
    return N, denominator


def difference_bounds(h, inverse):
    # For these expanding examples, rho < 1 and this box contains all ancestors.
    N, denominator = inverse
    m = len(N)
    rho = Rational(max(sum(abs(N[i][j]) for i in range(m))
                       for j in range(m)), denominator)
    C = int(ceiling(2*max(map(len, h.values()))*rho/(1-rho)))
    return (C,)*m


def difference_vectors(v, inverse, bounds, smith=None, condition=None,
                       known_solvable=False):
    m = len(v)
    C = bounds(v) if callable(bounds) else bounds
    if inverse is not None:
        N, denominator = inverse
        numerator = tuple(sum(v[i]*N[i][j] for i in range(m)) for j in range(m))
        if not known_solvable and any(x % denominator for x in numerator):
            return ()
        D = tuple(x // denominator for x in numerator)
        if all(abs(D[j]) <= C[j] for j in range(m)):
            if condition is None or condition(D):
                return (D,)
        return ()

    diagonal, L, R, U = smith
    r = sum(x != 0 for x in diagonal)
    w = tuple(sum(v[i]*R[i][j] for i in range(m)) for j in range(m))
    if not known_solvable:
        if any(w[j] % abs(diagonal[j]) for j in range(r)):
            return ()
        if any(w[j] for j in range(r, m)):
            return ()

    fixed = tuple(w[j] // diagonal[j] for j in range(r))
    free_bounds = [sum(C[i]*abs(U[i][j]) for i in range(m)) for j in range(r, m)]
    solutions = []
    for z in product(*(range(-int(b), int(b)+1) for b in free_bounds)):
        y = fixed+z
        D = tuple(sum(y[i]*L[i][j] for i in range(m)) for j in range(m))
        if all(abs(D[j]) <= C[j] for j in range(m)):
            if condition is None or condition(D):
                solutions.append(D)
    return tuple(solutions)


def parent_rhs(d, left, middle, right):
    return tuple(d[j]+left[2][j]+middle[1][j]-middle[2][j]-right[1][j]
                 for j in range(len(d)))


def score_based_parent_search(t, B, alphabet, weights, moduli,
                              inverse, bounds, smith=None, condition=None):
    A, D = t
    parents = set()
    chosen = []
    differences = []

    def lift(j):
        if j == len(A):
            parents.add((tuple(split[0] for split in chosen), tuple(differences)))
            return
        for split in buckets[j]:
            chosen.append(split)
            if j < 2:
                lift(j+1)
            else:
                v = parent_rhs(D[j-2], *chosen[-3:])
                for d in difference_vectors(v, inverse, bounds, smith, condition,
                                            known_solvable=True):
                    differences.append(d)
                    lift(j+1)
                    differences.pop()
            chosen.pop()

    for scores in admissible_score_sequences(t, B, alphabet, weights, moduli):
        buckets = [B[a][q] for a, q in zip(A, scores)]
        lift(0)
    return parents


def sieve2(t, alphabet):
    # True means keep. False means discard as non-genuine.
    A, D = t
    k = len(A)-1
    P = [parikh(a, alphabet) for a in A]
    nonempty = any(any(d) for d in D) or any(A[1:-1])
    states = {(x, y, bool(nonempty or (x == 0 and A[0] != '')))
              for x, y in product((0, 1), repeat=2)}
    for i in range(1, k):
        following = set()
        for x, y, nonempty in states:
            for z in (0, 1):
                difference = tuple(D[i-1][j]-(1-x)*P[i-1][j]
                                   +(1-2*y)*P[i][j]+z*P[i+1][j]
                                   for j in range(len(alphabet)))
                if not any(difference):
                    following.add((y, z, nonempty))
        states = following
        if not states:
            return True
    return not any(nonempty or (z == 1 and A[k] != '')
                   for y, z, nonempty in states)


def ancestor_search(t, parent_search, alphabet, use_sieve2=False):
    # The starting template is included. For Sieve 2, start from T_k.
    ancestors = {t}
    waiting = deque([t])
    while waiting:
        child = waiting.popleft()
        for parent in parent_search(child):
            if use_sieve2 and not sieve2(parent, alphabet):
                continue
            if parent not in ancestors:
                ancestors.add(parent)
                waiting.append(parent)
    return ancestors


# ==================== Search bounds ====================

def classic_search_bound(h, templates):
    k = len(templates[0][0])-1
    m = len(h)
    N = max(map(len, h.values()))
    Delta = isqrt(max(sum(x*x for x in d) for A, D in templates for d in D))
    bound = N+k-1+(k-1)*(N-2+m*k*Delta)
    return bound, Delta


def new_search_bound(h, templates):
    k = len(templates[0][0])-1
    N = max(map(len, h.values()))
    Omega = 0
    for A, D in templates:
        lengths = [0]
        for d in D:
            lengths.append(lengths[-1]+sum(d))
        Omega = max(Omega, max(lengths)-min(lengths))
    bound = N+k-1+(k-1)*(N-2+Omega)
    return bound, Omega


# ==================== Word length modulo k ====================

def word_length_modulo(t):
    A, D = t
    k = len(A)-1
    return (sum(map(len, A))
            + sum((k-1-i)*sum(d) for i, d in enumerate(D))) % k


# ==================== Parikh vector modulo k ====================

def parikh_modulo(t, alphabet):
    A, D = t
    k = len(A)-1
    boundary = parikh(''.join(A), alphabet)
    return tuple((boundary[j]
                  + sum((k-1-i)*d[j] for i, d in enumerate(D))) % k
                 for j in range(len(alphabet)))


# ==================== Complete distinct factors ====================

def factors_of_length(h, start, length):
    # Stop when two consecutive iterates have the same length-factor set.
    # First reach length >= 'length', so equality of two empty sets cannot stop us.
    # For non-erasing h, F_L(h(w)) depends only on F_L(w) when |w| >= L.
    word = start
    while len(word) < length:
        word = ''.join(h[a] for a in word)
    factors = {word[i:i+length] for i in range(len(word)-length+1)}
    while True:
        next_word = ''.join(h[a] for a in word)
        next_factors = {next_word[i:i+length]
                        for i in range(len(next_word)-length+1)}
        if next_factors == factors:
            return factors
        word, factors = next_word, next_factors


def distinct_factors(h, start, bound):
    # Store all distinct nonempty factors of lengths 1,...,bound in one list.
    # Every shorter factor extends to a length-bound factor in the fixed point.
    longest = factors_of_length(h, start, bound)
    factors = {word[:n] for word in longest for n in range(1, bound+1)}
    return sorted(factors, key=lambda word: (len(word), word))


# ==================== Count the four search stages ====================

def compare_search(h, templates, factors, new_bound):
    # 'factors' is the complete list up to the global classic bound.
    # The new bound and both modulo conditions are applied to this list.
    # No extra boundary or block-feasibility conditions are imposed.
    alphabet = tuple(h)
    k = len(templates[0][0])-1
    classic_count = len(factors)
    new_factors = [word for word in factors if len(word) <= new_bound]
    new_count = len(new_factors)
    length_counts = Counter(len(word) % k for word in new_factors)
    parikh_counts = Counter(tuple(x % k for x in parikh(word, alphabet))
                            for word in new_factors)

    length_residues = [word_length_modulo(t) for t in templates]
    parikh_residues = [parikh_modulo(t, alphabet) for t in templates]

    # Union: a factor is counted once if at least one template retains it.
    distinct = (classic_count, new_count,
                sum(length_counts[r] for r in set(length_residues)),
                sum(parikh_counts[r] for r in set(parikh_residues)))
    # Pairs: a factor is counted once for each template retaining it.
    pairs = (len(templates)*classic_count, len(templates)*new_count,
             sum(length_counts[r] for r in length_residues),
             sum(parikh_counts[r] for r in parikh_residues))
    return distinct, pairs


def display_table(title, rows, columns):
    print()
    display(Markdown('**'+title+'**'))
    text = ['| '+' | '.join(columns)+' |',
            '| '+' | '.join(['---']*len(columns))+' |']
    for row in rows:
        text.append('| '+' | '.join(f'{x:,}' if isinstance(x, int) else str(x)
                                   for x in row)+' |')
    display(Markdown('\n'.join(text)))


# ==================== Applications ====================

# Sources for the classic bound (the corrected coefficient is k-1):
# https://arxiv.org/pdf/1106.1842
# https://nrampers.faculty.uwinnipeg.ca/errata.html
# The new bound is Lemma 11.1; modulo conditions are Propositions 11.2--11.3.
display(Markdown('**Search bounds**'))
display(Math(r'\Delta=\left\lfloor\max_{t\in\mathcal T}\max_{1\le i<k}'
             r'\lVert d_i\rVert_2\right\rfloor,\qquad'
             r'\Omega=\max_{t\in\mathcal T}\max_{1\le i<j\le k}'
             r'\left|\sum_{r=i}^{j-1}d_r\cdot(1,\ldots,1)\right|'))
display(Math(r'B_{\mathrm{classic}}=N+k-1+(k-1)(N-2+mk\Delta)'))
display(Math(r'B_{\mathrm{new}}=N+k-1+(k-1)(N-2+\Omega)'))

results = []


# ==================== T8 ====================

name = 'T8'
h = {'a': 'abaaaba', 'b': 'babab'}
k = 8
weights = {'': (0,), 'a': (1,), 'b': (3,)}
moduli = (11,)
alphabet = ('a', 'b')
t = zero_template(k, alphabet)
M = frequency_matrix(h, alphabet)
inverse = prepare_inverse(M)
bounds = difference_bounds(h, inverse)

S = split_sets(h, alphabet)
B = score_buckets(S, alphabet, weights, moduli)
B1 = nontrivial_buckets(B)


def parents1(child):
    return score_based_parent_search(
        child, B1, alphabet, weights, moduli, inverse, bounds)

templates = sorted(ancestor_search(t, parents1, alphabet, use_sieve2=True))

classic, Delta = classic_search_bound(h, templates)
new, Omega = new_search_bound(h, templates)
N, m = max(map(len, h.values())), len(alphabet)

print()
display(Markdown('**'+name+' — '+str(len(templates))+' sieved ancestors**'))
print_morphism(h, k)
display(Math(rf'N={N},\quad m={m},\quad k={k},\quad '
             rf'\Delta={Delta},\quad\Omega={Omega}'))
display(Math(rf'B_{{\mathrm{{classic}}}}={N}+{k}-1+'
             rf'({k}-1)({N}-2+{m}\cdot{k}\cdot{Delta})={classic}'))
display(Math(rf'B_{{\mathrm{{new}}}}={N}+{k}-1+'
             rf'({k}-1)({N}-2+{Omega})={new}'))

factors = distinct_factors(h, 'a', classic)
distinct, pairs = compare_search(h, templates, factors, new)
results.append({'name': name, 'h': h, 'k': k, 'templates': templates,
                'classic_bound': classic, 'new_bound': new,
                'factors': factors,
                'distinct_factors': distinct, 'factor_template_pairs': pairs})


# ==================== Andrade-Mol ====================

name = 'Andrade-Mol'
h = {'a': 'aaaab', 'b': 'abbab'}
k = 5
weights = {'': (0,), 'a': (1,), 'b': (6,)}
moduli = (10,)
alphabet = ('a', 'b')
t = zero_template(k, alphabet)
M = frequency_matrix(h, alphabet)
inverse = prepare_inverse(M)
bounds = difference_bounds(h, inverse)

S = split_sets(h, alphabet)
B = score_buckets(S, alphabet, weights, moduli)
B1 = nontrivial_buckets(B)


def parents1(child):
    return score_based_parent_search(
        child, B1, alphabet, weights, moduli, inverse, bounds)

templates = sorted(ancestor_search(t, parents1, alphabet, use_sieve2=True))

classic, Delta = classic_search_bound(h, templates)
new, Omega = new_search_bound(h, templates)
N, m = max(map(len, h.values())), len(alphabet)

print()
display(Markdown('**'+name+' — '+str(len(templates))+' sieved ancestors**'))
print_morphism(h, k)
display(Math(rf'N={N},\quad m={m},\quad k={k},\quad '
             rf'\Delta={Delta},\quad\Omega={Omega}'))
display(Math(rf'B_{{\mathrm{{classic}}}}={N}+{k}-1+'
             rf'({k}-1)({N}-2+{m}\cdot{k}\cdot{Delta})={classic}'))
display(Math(rf'B_{{\mathrm{{new}}}}={N}+{k}-1+'
             rf'({k}-1)({N}-2+{Omega})={new}'))

factors = distinct_factors(h, 'a', classic)
distinct, pairs = compare_search(h, templates, factors, new)
results.append({'name': name, 'h': h, 'k': k, 'templates': templates,
                'classic_bound': classic, 'new_bound': new,
                'factors': factors,
                'distinct_factors': distinct, 'factor_template_pairs': pairs})


# ==================== T5 ====================

name = 'T5'
h = {'a': 'aaaab', 'b': 'ababab'}
k = 5
weights = {'': (0,), 'a': (1,), 'b': (5,)}
moduli = (9,)
alphabet = ('a', 'b')
t = zero_template(k, alphabet)
M = frequency_matrix(h, alphabet)
inverse = prepare_inverse(M)
bounds = difference_bounds(h, inverse)

S = split_sets(h, alphabet)
B = score_buckets(S, alphabet, weights, moduli)
B1 = nontrivial_buckets(B)


def parents1(child):
    return score_based_parent_search(
        child, B1, alphabet, weights, moduli, inverse, bounds)

templates = sorted(ancestor_search(t, parents1, alphabet, use_sieve2=True))

classic, Delta = classic_search_bound(h, templates)
new, Omega = new_search_bound(h, templates)
N, m = max(map(len, h.values())), len(alphabet)

print()
display(Markdown('**'+name+' — '+str(len(templates))+' sieved ancestors**'))
print_morphism(h, k)
display(Math(rf'N={N},\quad m={m},\quad k={k},\quad '
             rf'\Delta={Delta},\quad\Omega={Omega}'))
display(Math(rf'B_{{\mathrm{{classic}}}}={N}+{k}-1+'
             rf'({k}-1)({N}-2+{m}\cdot{k}\cdot{Delta})={classic}'))
display(Math(rf'B_{{\mathrm{{new}}}}={N}+{k}-1+'
             rf'({k}-1)({N}-2+{Omega})={new}'))

factors = distinct_factors(h, 'a', classic)
distinct, pairs = compare_search(h, templates, factors, new)
results.append({'name': name, 'h': h, 'k': k, 'templates': templates,
                'classic_bound': classic, 'new_bound': new,
                'factors': factors,
                'distinct_factors': distinct, 'factor_template_pairs': pairs})


# ==================== T14 ====================

name = 'T14'
h = {'a': 'abaabaababa', 'b': 'babababab'}
k = 14
weights = {'': (0,), 'a': (1,), 'b': (3,)}
moduli = (19,)
alphabet = ('a', 'b')
t = zero_template(k, alphabet)
M = frequency_matrix(h, alphabet)
inverse = prepare_inverse(M)
bounds = difference_bounds(h, inverse)

S1 = split_sets(h, alphabet, nontrivial=True)
B1 = score_buckets(S1, alphabet, weights, moduli)


def parents1(child):
    return score_based_parent_search(
        child, B1, alphabet, weights, moduli, inverse, bounds)

templates = sorted(ancestor_search(t, parents1, alphabet, use_sieve2=True))

classic, Delta = classic_search_bound(h, templates)
new, Omega = new_search_bound(h, templates)
N, m = max(map(len, h.values())), len(alphabet)

print()
display(Markdown('**'+name+' — '+str(len(templates))+' sieved ancestors**'))
print_morphism(h, k)
display(Math(rf'N={N},\quad m={m},\quad k={k},\quad '
             rf'\Delta={Delta},\quad\Omega={Omega}'))
display(Math(rf'B_{{\mathrm{{classic}}}}={N}+{k}-1+'
             rf'({k}-1)({N}-2+{m}\cdot{k}\cdot{Delta})={classic}'))
display(Math(rf'B_{{\mathrm{{new}}}}={N}+{k}-1+'
             rf'({k}-1)({N}-2+{Omega})={new}'))

factors = distinct_factors(h, 'a', classic)
distinct, pairs = compare_search(h, templates, factors, new)
results.append({'name': name, 'h': h, 'k': k, 'templates': templates,
                'classic_bound': classic, 'new_bound': new,
                'factors': factors,
                'distinct_factors': distinct, 'factor_template_pairs': pairs})


# Final tables
display_table('Bounds and sieved ancestors',
              [(r['h']['a'], r['h']['b'], r['k'], len(r['templates']),
                r['classic_bound'], r['new_bound']) for r in results],
              ['h(a)', 'h(b)', 'k', 'Ancestors', 'Classic bound', 'New bound'])

display_table('Distinct factors: union over templates',
              [(r['h']['a'], r['h']['b'], r['k'], *r['distinct_factors']) for r in results],
              ['h(a)', 'h(b)', 'k', 'Classic', 'New', '+ length mod k', '+ Parikh mod k'])

display_table('Factor-template pairs',
              [(r['h']['a'], r['h']['b'], r['k'], *r['factor_template_pairs']) for r in results],
              ['h(a)', 'h(b)', 'k', 'Classic', 'New', '+ length mod k', '+ Parikh mod k'])


**Search bounds**

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

**T8 — 111 sieved ancestors**

<IPython.core.display.Math object>

... (20 outputs left out)

**Distinct factors: union over templates**

| h(a) | h(b) | k | Classic | New | + length mod k | + Parikh mod k |
| --- | --- | --- | --- | --- | --- | --- |
| abaaaba | babab | 8 | 30,389 | 3,803 | 3,370 | 1,274 |
| aaaab | abbab | 5 | 574 | 574 | 262 | 52 |
| aaaab | ababab | 5 | 10,767 | 868 | 717 | 315 |
| abaabaababa | babababab | 14 | 230,965 | 26,629 | 23,124 | 4,898 |

**Factor-template pairs**

| h(a) | h(b) | k | Classic | New | + length mod k | + Parikh mod k |
| --- | --- | --- | --- | --- | --- | --- |
| abaaaba | babab | 8 | 3,373,179 | 422,133 | 52,879 | 7,765 |
| aaaab | abbab | 5 | 1,722 | 1,722 | 399 | 78 |
| aaaab | ababab | 5 | 1,216,671 | 98,084 | 20,192 | 4,496 |
| abaabaababa | babababab | 14 | 227,500,525 | 26,229,565 | 1,886,005 | 148,881 |